In [1]:
%%bash
export PYTHONPATH=""
CONDA_DIR="/kaggle/working/miniconda"
ENV_DIR="${CONDA_DIR}/envs/py310"
PIP="${ENV_DIR}/bin/pip"
REPO_DIR="/kaggle/working/MCoT-MVS-Online"
WEIGHTS_DIR="/kaggle/input/datasets/b22dckh099lngminhqu/mcot-mvs-weights/weights"

echo "=========================================================="
echo "BƯỚC 1: KHỞI TẠO MINICONDA & PYTHON 3.10"
echo "=========================================================="
if [ ! -d "$CONDA_DIR" ]; then
    echo "• Tải và cài đặt Miniconda..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    bash miniconda.sh -b -p $CONDA_DIR -u > /dev/null
    rm miniconda.sh
    $CONDA_DIR/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main > /dev/null 2>&1 || true
    $CONDA_DIR/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r > /dev/null 2>&1 || true
fi

if [ ! -d "$ENV_DIR" ]; then
    echo "• Tạo môi trường py310 (Python 3.10)..."
    $CONDA_DIR/bin/conda create -n py310 -c conda-forge python=3.10 -y -q
fi

echo "=========================================================="
echo "BƯỚC 2: CLONE MÃ NGUỒN GỐC DỰ ÁN"
echo "=========================================================="
cd /kaggle/working
if [ ! -d "MCoT-MVS-Online" ]; then
    git clone https://github.com/LMQ-Ptit/MCoT-MVS-Online.git
else
    cd MCoT-MVS-Online && git pull && cd ..
fi

echo "=========================================================="
echo "BƯỚC 3: CÀI ĐẶT PYTORCH 2.3.1 (CUDA 12.1) & DEPENDENCIES"
echo "=========================================================="
echo "• Cài PyTorch 2.3.1..."
$PIP install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121

echo "• Cài thư viện gốc (requirements.txt)..."
$PIP install -q -r ${REPO_DIR}/requirements.txt

echo "• Cài thư viện tương thích cho Segment & Qwen2.5-VL..."
$PIP install -q \
    transformers==4.49.0 \
    accelerate==1.4.0 \
    qwen-vl-utils==0.0.8 \
    timm==0.9.16 \
    fairscale==0.4.13 \
    pycocotools \
    git+https://github.com/xinyu1205/recognize-anything.git
    
echo "=========================================================="
echo "BƯỚC 4: LIÊN KẾT TRỌNG SỐ TỪ DATASET KAGGLE"
echo "=========================================================="
mkdir -p ${REPO_DIR}/segment/sam2/checkpoints
mkdir -p ${REPO_DIR}/segment/ram_models
mkdir -p ${REPO_DIR}/src/checkpoints

SAM_SRC=$(find /kaggle/input -name "sam2.1_hiera_large.pt" | head -n 1)
RAM_SRC=$(find /kaggle/input -name "*ram_plus_swin_large*" | head -n 1)
DRESS_SRC=$(find /kaggle/input -name "dress.pt" | head -n 1)
SHIRT_SRC=$(find /kaggle/input -name "shirt.pt" | head -n 1)

ln -sf "$SAM_SRC" ${REPO_DIR}/segment/sam2/checkpoints/sam2.1_hiera_large.pt
ln -sf "$RAM_SRC" ${REPO_DIR}/segment/ram_models/ram_plus_swin_large_14m.pth
ln -sf "$DRESS_SRC" ${REPO_DIR}/src/checkpoints/dress.pt
ln -sf "$SHIRT_SRC" ${REPO_DIR}/src/checkpoints/shirt.pt

echo "Kiểm tra liên kết trọng số:"
ls -lhL ${REPO_DIR}/segment/sam2/checkpoints/sam2.1_hiera_large.pt
ls -lhL ${REPO_DIR}/segment/ram_models/ram_plus_swin_large_14m.pth
ls -lhL ${REPO_DIR}/src/checkpoints/dress.pt
ls -lhL ${REPO_DIR}/src/checkpoints/shirt.pt

echo "=========================================================="
echo "🎉 SETUP HOÀN TẤT 100%! HỆ THỐNG ĐÃ SẴN SÀNG."
echo "=========================================================="

BƯỚC 1: KHỞI TẠO MINICONDA & PYTHON 3.10
• Tải và cài đặt Miniconda...
• Tạo môi trường py310 (Python 3.10)...
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/py310

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_103         725 KB  conda-forge
    libexpat-2.9.0             |       hd2095e1_0          77 KB  conda-forge
    libffi-3.7.0               | 

WARNING conda.conda_pypi.main:notify_conda_pypi_tip(158): 
  Did you know? You can install many PyPI packages with conda
  using conda-pypi. Get started:
    https://bit.ly/4xXYt0B

Cloning into 'MCoT-MVS-Online'...


In [2]:
%%script /kaggle/working/miniconda/envs/py310/bin/python
import sys
print("Phiên bản hiện tại đang chạy:")
print(sys.version)

# Viết code của bạn ở đây

Phiên bản hiện tại đang chạy:
3.10.22 | packaged by conda-forge | (main, Oct  6 2026, 23:02:24) [GCC 14.4.0]


In [4]:
!cd /kaggle/working/MCoT-MVS-Online && git pull

Already up to date.


In [5]:
!/kaggle/working/miniconda/envs/py310/bin/python -c "import transformers; print('Transformers version:', transformers.__version__)"

Transformers version: 4.49.0


In [4]:
!/kaggle/working/miniconda/envs/py310/bin/python -c "import torch; print('GPU Có sẵn:', torch.cuda.is_available(), '| Số lượng GPU:', torch.cuda.device_count())"

GPU Có sẵn: True | Số lượng GPU: 2


In [3]:
%%script /kaggle/working/miniconda/envs/py310/bin/python
import sys
sys.path.append("/kaggle/working/MCoT-MVS-Online/segment")

from get_masks import load_segment_pipeline

# Khởi tạo pipeline
pipeline = load_segment_pipeline(device="cuda:0", load_clip=True)

# Test inference ảnh online mẫu
test_image = "https://raw.githubusercontent.com/open-mmlab/mmdetection/master/demo/demo.jpg"
result = pipeline.process_image(
    image_input=test_image,
    save_dir="/kaggle/working/infer_output",
    is_draw=True,
    compute_clip_features=True
)

print("\n🎉 KẾT QUẢ INFERENCE THÀNH CÔNG:")
print("  • Tags phát hiện được  :", result["tags"])
print("  • Bounding boxes       :", result["boxes"].shape)
print("  • Instance Masks       :", result["masks"].shape)
print("  • Segment Features     :", result["seg_features"].shape)
print("  • Ảnh kết quả lưu tại  :", result["annotated_image_path"])

⏳ Loading Segment Pipeline on [cuda:0]...
--------------
/kaggle/working/MCoT-MVS-Online/segment/ram_models/ram_plus_swin_large_14m.pth
--------------
load checkpoint from /kaggle/working/MCoT-MVS-Online/segment/ram_models/ram_plus_swin_large_14m.pth
vit: swin_l
⏳ Loading OpenCLIP ViT-H-14 for mask feature pooling...
✅ Segment Pipeline loaded successfully!

🎉 KẾT QUẢ INFERENCE THÀNH CÔNG:
  • Tags phát hiện được  : ['area', 'bench', 'black', 'car', 'parking lot', 'field', 'grass', 'grassy', 'green', 'lawn', 'lush', 'park', 'park bench', 'pavement', 'road', 'seat', 'shadow', 'sit', 'tree']
  • Bounding boxes       : (6, 4)
  • Instance Masks       : (6, 427, 640)
  • Segment Features     : torch.Size([6, 1024])
  • Ảnh kết quả lưu tại  : /kaggle/working/infer_output/grounded_sam2_annotated_image_with_mask.jpg


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [5]:
%%script /kaggle/working/miniconda/envs/py310/bin/python
import sys
import json
from PIL import Image
import urllib.request
sys.path.append("/kaggle/working/MCoT-MVS-Online/src")

from vllm_extractor import MCoTExtractor

# 1. Khởi tạo extractor trên GPU 1 (cuda:1)
extractor = MCoTExtractor(
    mode="local",
    model_name="Qwen/Qwen2.5-VL-3B-Instruct",
    device="cuda:1"
)

# 2. Ảnh mẫu test và câu lệnh thay đổi
test_img_url = "https://raw.githubusercontent.com/open-mmlab/mmdetection/master/demo/demo.jpg"
image = Image.open(urllib.request.urlopen(test_img_url)).convert("RGB")
text_modification = "is darker and has short sleeves with casual fit"

# 3. Trích xuất MCoT reasoning
print("\n⏳ Đang trích xuất MCoT reasoning từ Qwen2.5-VL-3B...")
result = extractor.extract(image, text_modification)

print("\n🎉 KẾT QUẢ MCoT EXTRACTION THÀNH CÔNG:")
print(json.dumps(result, indent=2, ensure_ascii=False))

⏳ Loading local VLLM model: [Qwen/Qwen2.5-VL-3B-Instruct] on [cuda:1]...
✅ VLLM Model loaded successfully on GPU [cuda:1]! VRAM used: 7276.7 MB

⏳ Đang trích xuất MCoT reasoning từ Qwen2.5-VL-3B...

🎉 KẾT QUẢ MCoT EXTRACTION THÀNH CÔNG:
{
  "target": "a person wearing a dark-colored shirt with short sleeves and a casual fit",
  "retained": "the color of the shirt and the casual fit",
  "deleted": "any other clothing items or accessories"
}


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.65s/it]


In [8]:
%%script /kaggle/working/miniconda/envs/py310/bin/python
"""
NHIỆM VỤ 1 : GHÉP NỐI VLLM + SEGMENT + CIRMODEL
- cuda:0 : Segment Pipeline (RAM++ + DINO + SAM2)
- cuda:1 : MCoT VLLM Extractor (Qwen2.5-VL-3B) + CIRModel (float32)
"""
import sys, time, json, os
import torch
import torch.nn.functional as F
from PIL import Image
import urllib.request

sys.path.append("/kaggle/working/MCoT-MVS-Online/segment")
sys.path.append("/kaggle/working/MCoT-MVS-Online/src")

from get_masks import load_segment_pipeline
from vllm_extractor import MCoTExtractor
from model import CIRModel

REPO_DIR = "/kaggle/working/MCoT-MVS-Online"
DRESS_TYPE = "dress"  # hoặc 'shirt'

def log_vram(tag=""):
    print(f"--- VRAM {tag} ---")
    for i in range(torch.cuda.device_count()):
        used = torch.cuda.memory_allocated(i) / (1024**3)
        res  = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"  GPU {i} (cuda:{i}): Used={used:.2f}GB | Reserved={res:.2f}GB / 15.0GB")

# ============================================================
# 1. NẠP CÁC MODEL VÀO ĐÚNG GPU
# ============================================================
print("=" * 60)
print("🚀 [1/3] Nạp Segment Pipeline lên GPU 0 (cuda:0)...")
t0 = time.time()
torch.cuda.set_device(0)  # Đảm bảo RAM++ và SAM2 nằm hoàn toàn ở cuda:0
seg_pipeline = load_segment_pipeline(device="cuda:0", load_clip=True)
print(f"✅ Segment Pipeline đã sẵn sàng! ({time.time()-t0:.1f}s)")
log_vram("Sau Segment")

print("\n🚀 [2/3] Nạp MCoT VLLM Extractor lên GPU 1 (cuda:1)...")
t0 = time.time()
extractor = MCoTExtractor(
    mode="local",
    model_name="Qwen/Qwen2.5-VL-3B-Instruct",
    device="cuda:1"
)
print(f"✅ MCoT VLLM đã sẵn sàng! ({time.time()-t0:.1f}s)")
log_vram("Sau VLLM")

print(f"\n🚀 [3/3] Nạp CIRModel ({DRESS_TYPE}.pt) lên GPU 1 (cuda:1)...")
t0 = time.time()
torch.cuda.set_device(1)  # Đặt ngữ cảnh cuda:1 cho CIRModel
cir_model = CIRModel()

ckpt_path = f"{REPO_DIR}/src/checkpoints/{DRESS_TYPE}.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")
if isinstance(ckpt, dict):
    if "CIRModel" in ckpt:     state_dict = ckpt["CIRModel"]
    elif "state_dict" in ckpt:  state_dict = ckpt["state_dict"]
    elif "model" in ckpt:       state_dict = ckpt["model"]
    else:                       state_dict = ckpt
else:
    state_dict = ckpt

cir_model.load_state_dict(state_dict, strict=False)
cir_model = cir_model.to("cuda:1").eval()  # Chạy float32 chuẩn gốc
print(f"✅ CIRModel đã nạp thành công lên cuda:1! ({time.time()-t0:.1f}s)")
log_vram("Sau khi nạp đủ 3 mô hình")

# ============================================================
# 2. HÀM PIPELINE ONLINE TÍCH HỢP HOÀN CHỈNH
# ============================================================
def extract_composed_query(reference_image_input, modified_caption: str):
    """
    Quy trình MCoT-MVS Online:
    Ảnh gốc + Text thay đổi -> [Segment + VLLM] -> [CIRModel] -> Query Vector (1, 1024)
    """
    if isinstance(reference_image_input, str):
        if reference_image_input.startswith("http"):
            pil_img = Image.open(urllib.request.urlopen(reference_image_input)).convert("RGB")
        else:
            pil_img = Image.open(reference_image_input).convert("RGB")
        img_input = reference_image_input
    else:
        pil_img = reference_image_input
        img_input = reference_image_input

    # --- Bước A: Phân vùng trên GPU 0 (cuda:0) ---
    torch.cuda.set_device(0)  # ★ FIX QUAN TRỌNG: Trả về cuda:0 để RAM++ không bị lỗi
    seg_res = seg_pipeline.process_image(
        image_input=img_input,
        save_dir="/kaggle/working/infer_output",
        is_draw=False,
        compute_clip_features=True
    )
    # Chuyển vector phân vùng sang cuda:1 để phục vụ CIRModel
    seg_features = seg_res["seg_features"].to("cuda:1")

    # --- Bước B: MCoT VLLM suy luận trên GPU 1 (cuda:1) ---
    mcot_info = extractor.extract(pil_img, modified_caption)

    # --- Bước C: CIRModel dung hợp trên GPU 1 (cuda:1) ---
    torch.cuda.set_device(1)  # Chuyển về cuda:1 để CIRModel tính toán
    with torch.no_grad():
        query_vector = cir_model.extract_query(
            modified_text=[modified_caption],
            reference_image=[pil_img],
            reference_seg_feature_list=[seg_features],
            llm_info_dict={
                "target":   [mcot_info["target"]],
                "retained": [mcot_info["retained"]],
                "deleted":  [mcot_info["deleted"]],
            },
        )
    return query_vector, mcot_info, seg_features

# ============================================================
# 3. WARM-UP MÔ HÌNH (Chạy mẫu 1 lượt)
# ============================================================
print("\n" + "=" * 60)
print("🔥 ĐANG WARM-UP TOÀN BỘ HỆ THỐNG...")
print("=" * 60)
warmup_url = "https://raw.githubusercontent.com/open-mmlab/mmdetection/master/demo/demo.jpg"
warmup_cap = "is blue with white dots"

t_warmup = time.time()
q_vec, mcot, sf = extract_composed_query(warmup_url, warmup_cap)
print(f"🎉 Warm-up hoàn tất thành công trong {time.time()-t_warmup:.2f}s!")
print(f"  • MCoT Target   : {mcot['target']}")
print(f"  • MCoT Retained : {mcot['retained']}")
print(f"  • MCoT Deleted  : {mcot['deleted']}")
print(f"  • seg_features  : {sf.shape} trên {sf.device}")
print(f"  • Vector Query  : shape={q_vec.shape}, dtype={q_vec.dtype}, norm={q_vec.norm().item():.4f}")
print("=" * 60)

🚀 [1/3] Nạp Segment Pipeline lên GPU 0 (cuda:0)...
⏳ Loading Segment Pipeline on [cuda:0]...
--------------
/kaggle/working/MCoT-MVS-Online/segment/ram_models/ram_plus_swin_large_14m.pth
--------------
load checkpoint from /kaggle/working/MCoT-MVS-Online/segment/ram_models/ram_plus_swin_large_14m.pth
vit: swin_l
⏳ Loading OpenCLIP ViT-H-14 for mask feature pooling...
✅ Segment Pipeline loaded successfully!
✅ Segment Pipeline đã sẵn sàng! (31.7s)
--- VRAM Sau Segment ---
  GPU 0 (cuda:0): Used=6.62GB | Reserved=6.72GB / 15.0GB
  GPU 1 (cuda:1): Used=0.00GB | Reserved=0.00GB / 15.0GB

🚀 [2/3] Nạp MCoT VLLM Extractor lên GPU 1 (cuda:1)...
⏳ Loading local VLLM model: [Qwen/Qwen2.5-VL-3B-Instruct] on [cuda:1]...
✅ VLLM Model loaded successfully on GPU [cuda:1]! VRAM used: 7276.7 MB
✅ MCoT VLLM đã sẵn sàng! (25.8s)
--- VRAM Sau VLLM ---
  GPU 0 (cuda:0): Used=6.62GB | Reserved=6.72GB / 15.0GB
  GPU 1 (cuda:1): Used=7.11GB | Reserved=7.31GB / 15.0GB

🚀 [3/3] Nạp CIRModel (dress.pt) lên GPU 1 

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 2/2 [00:24<00:00, 12.10s/it]
